# Dates Generator -- Research-Backed Redesign

Conditional generation of dates given (DOW, MON, LEAP, DEC).
Four research-backed models: CVAE+CFG, AC-GAN (hybrid-MLE + WGAN-GP),
MaskGIT, MDLM+CFG.

**Designed for Google Colab GPU runtime.** Local Jupyter works too --
the first cell auto-detects the environment.

**Colab quickstart:**
1. Runtime -> Change runtime type -> T4 GPU.
2. Run cell 0 below. In Colab it will prompt for a GitHub Personal
   Access Token (PAT) to clone the private repo into `/content/submission`.
3. Run all remaining cells.


## 0. Environment setup (Colab clone + deps install, or local)


In [13]:
# Detect Colab; clone the public repo; install deps; cd into the repo root.
import os, sys, subprocess
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print('Colab:', IN_COLAB)

# === EDIT THESE for your fork ===
GITHUB_USER = 'ustafaa'
REPO_NAME   = 'dates-generator'
BRANCH      = 'main'

if IN_COLAB:
    repo_dir = f'/content/{REPO_NAME}'
    if not os.path.isdir(repo_dir):
        url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
        subprocess.check_call(['git', 'clone', '-b', BRANCH, url, repo_dir])
    os.chdir(repo_dir)
    print('cwd ->', os.getcwd())
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
else:
    print('cwd ->', os.getcwd())

if '.' not in sys.path:
    sys.path.insert(0, '.')

import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


Colab: True
cwd -> /content/dates-generator
torch 2.10.0+cu128 cuda True Tesla T4


## 1. Load data + verify splits


In [14]:
from model.data import load_records, build_splits, build_held_out_tuples
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
records = load_records('data/data.txt')
in_set, held = build_held_out_tuples(records, n_held_out=50, seed=0)
train, val, test = build_splits(in_set, val_frac=0.1, test_frac=0.1, seed=0)
print(f'records={len(records):,} train={len(train):,} val={len(val):,} test={len(test):,} held={len(held):,}')


records=146,462 train=116,279 val=14,534 test=14,534 held=1,115


## 2. Baselines: CSR floor and rejection-sampling ceiling


In [15]:
from model.evaluation import baseline_random, baseline_smart_random
print('random      :', baseline_random(val[:2000], seed=0))
print('smart_random:', baseline_smart_random(val[:2000], seed=0))


random      : n=2000    valid=1.000  CSR_all=0.085  DOW=0.141  MON=1.000  LEAP=0.635  DEC=1.000
smart_random: n=2000    valid=1.000  CSR_all=1.000  DOW=1.000  MON=1.000  LEAP=1.000  DEC=1.000


## 3. Train the four models

Each model takes ~10-30 min on a T4. Skip and load pretrained weights
if `model/weights/*.pt` already exists.


In [16]:
import pathlib
WEIGHTS = pathlib.Path('model/weights')
needs_train = not all((WEIGHTS / f'{n}.pt').exists() for n in ['cvae', 'acgan', 'maskgit', 'mdlm'])
print('needs_train:', needs_train)

if needs_train:
    !python model/train.py --data data/data.txt --out model/weights \
        --models cvae acgan maskgit mdlm \
        --epochs-cvae 15 --epochs-acgan 25 --epochs-maskgit 15 --epochs-mdlm 15 \
        --batch-size 1024 --seed 0
else:
    print('Pretrained weights found; skipping training.')


needs_train: True
[train.py] Loading data/data.txt
[train.py] 146,462 records
[train.py] splits: train=116,279  val=14,534  test=14,534  held_out_tuples=1,115
[cvae] params: 888,758
[cvae] ep 0 loss=5.5908 val_CSR=0.145
[cvae] ep 1 loss=5.4022 val_CSR=0.147
[cvae] ep 2 loss=5.5964 val_CSR=0.146
[cvae] ep 3 loss=5.5741 val_CSR=0.155
[cvae] ep 4 loss=5.5189 val_CSR=0.220
[cvae] ep 5 loss=5.0168 val_CSR=0.706
[cvae] ep 6 loss=4.2836 val_CSR=0.914
[cvae] ep 7 loss=4.0133 val_CSR=0.966
[cvae] ep 8 loss=3.8913 val_CSR=0.982
[cvae] ep 9 loss=3.8418 val_CSR=0.991
[cvae] ep10 loss=3.8118 val_CSR=0.994
[cvae] ep11 loss=3.7809 val_CSR=0.997
[cvae] ep12 loss=3.7620 val_CSR=0.998
[cvae] ep13 loss=3.7406 val_CSR=1.000
[cvae] ep14 loss=3.7317 val_CSR=1.000
[acgan] G=254,454 D=246,465
[acgan] ep 0 d=0.103 g=2.309 tau=1.00 val_CSR=0.108
[acgan] ep 1 d=-0.363 g=1.823 tau=0.97 val_CSR=0.109
[acgan] ep 2 d=-0.415 g=1.394 tau=0.94 val_CSR=0.133
[acgan] ep 3 d=-0.429 g=0.849 tau=0.91 val_CSR=0.128
[acgan] e

## 4. Load the four models and evaluate on val + held-out


In [17]:
from model.models.cvae import CVAE
from model.models.acgan import Generator, acgan_sample
from model.models.maskgit import MaskGIT
from model.models.mdlm import MDLM
from model.evaluation import evaluate_records

def load_one(name, cls, key='state_dict'):
    ck = torch.load(f'model/weights/{name}.pt', map_location=device, weights_only=False)
    m = cls(**ck['cfg']).to(device)
    m.load_state_dict(ck[key])
    return m.train(False)

models = {
    'cvae': load_one('cvae', CVAE),
    'acgan': load_one('acgan', Generator, key='state_dict_G'),
    'maskgit': load_one('maskgit', MaskGIT),
    'mdlm': load_one('mdlm', MDLM),
}

def make_gen(name, m):
    def gen(conds):
        idx = torch.tensor([list(c.as_indices()) for c in conds], dtype=torch.long, device=device)
        if name == 'acgan':
            d, yu = acgan_sample(m, idx, tau=0.3)
        else:
            d, yu = m.sample(idx, w=2.5)
        return [(int(d[i].item()) + 1, c.month_num, c.decade_int * 10 + int(yu[i].item()))
                for i, c in enumerate(conds)]
    return gen

for name, m in models.items():
    r = evaluate_records(make_gen(name, m), val[:2000])
    print(f'{name:10s} val          : {r}')
for name, m in models.items():
    r = evaluate_records(make_gen(name, m), held[:2000])
    print(f'{name:10s} held-out OOD : {r}')


/content/dates-generator/model/models/maskgit.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.backbone = nn.TransformerEncoder(layer, num_layers=n_layers)


cvae       val          : n=2000    valid=1.000  CSR_all=0.998  DOW=0.998  MON=1.000  LEAP=1.000  DEC=1.000
acgan      val          : n=2000    valid=0.998  CSR_all=0.149  DOW=0.149  MON=0.998  LEAP=0.995  DEC=0.998
maskgit    val          : n=2000    valid=1.000  CSR_all=0.144  DOW=0.144  MON=1.000  LEAP=1.000  DEC=1.000
mdlm       val          : n=2000    valid=0.999  CSR_all=0.994  DOW=0.994  MON=0.999  LEAP=0.999  DEC=0.999
cvae       held-out OOD : n=1115    valid=1.000  CSR_all=1.000  DOW=1.000  MON=1.000  LEAP=1.000  DEC=1.000
acgan      held-out OOD : n=1115    valid=0.999  CSR_all=0.131  DOW=0.132  MON=0.999  LEAP=0.997  DEC=0.999
maskgit    held-out OOD : n=1115    valid=1.000  CSR_all=0.139  DOW=0.139  MON=1.000  LEAP=1.000  DEC=1.000
mdlm       held-out OOD : n=1115    valid=1.000  CSR_all=0.987  DOW=0.987  MON=1.000  LEAP=1.000  DEC=1.000


## 5. Sample outputs and per-condition failure analysis


In [18]:
from model.metrics import check_dow, check_leap
from model.tokenizer import valid_date
for name, m in models.items():
    print(f'\n--- {name} ---')
    conds = [c for c, *_ in val[:6]]
    out = make_gen(name, m)(conds)
    for c, (d, mo, y) in zip(conds, out):
        ok_v = valid_date(d, mo, y)
        ok_dow = ok_v and check_dow(d, mo, y, c)
        ok_leap = ok_v and check_leap(d, mo, y, c)
        print(f'  {c.as_prefix()} -> {d}-{mo}-{y}  valid={ok_v} dow={ok_dow} leap={ok_leap}')



--- cvae ---
  [TUE] [APR] [False] [190] -> 30-4-1901  valid=True dow=True leap=True
  [SAT] [NOV] [False] [205] -> 27-11-2055  valid=True dow=True leap=True
  [THU] [DEC] [True] [216] -> 11-12-2160  valid=True dow=True leap=True
  [TUE] [OCT] [False] [201] -> 31-10-2017  valid=True dow=True leap=True
  [THU] [AUG] [True] [180] -> 16-8-1804  valid=True dow=True leap=True
  [WED] [AUG] [False] [200] -> 13-8-2003  valid=True dow=True leap=True

--- acgan ---
  [TUE] [APR] [False] [190] -> 15-4-1905  valid=True dow=False leap=True
  [SAT] [NOV] [False] [205] -> 8-11-2059  valid=True dow=True leap=True
  [THU] [DEC] [True] [216] -> 29-12-2168  valid=True dow=True leap=True
  [TUE] [OCT] [False] [201] -> 12-10-2011  valid=True dow=False leap=True
  [THU] [AUG] [True] [180] -> 6-8-1808  valid=True dow=False leap=True
  [WED] [AUG] [False] [200] -> 14-8-2003  valid=True dow=False leap=True

--- maskgit ---
  [TUE] [APR] [False] [190] -> 28-4-1906  valid=True dow=False leap=True
  [SAT] [NOV]

## 6. Final CSR table, ablation, and figures


In [19]:
!python scripts/run_final_eval.py
!python scripts/run_cfg_ablation.py
import pandas as pd
pd.read_csv('results/csr_table.csv')


/content/dates-generator/model/models/maskgit.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.backbone = nn.TransformerEncoder(layer, num_layers=n_layers)
wrote /content/dates-generator/results/csr_table.csv
wrote figures -> /content/dates-generator/results/figures
w=1.0  CSR=0.979  DOW=0.980  LEAP=0.998  div=0.717
w=1.5  CSR=0.995  DOW=0.995  LEAP=1.000  div=0.695
w=2.0  CSR=0.999  DOW=0.999  LEAP=1.000  div=0.673
w=2.5  CSR=0.999  DOW=0.999  LEAP=1.000  div=0.642
w=3.0  CSR=1.000  DOW=1.000  LEAP=1.000  div=0.618
w=4.0  CSR=1.000  DOW=1.000  LEAP=1.000  div=0.547
wrote /content/dates-generator/results/cfg_ablation.csv and figure


,model,split,csr_all,csr_dow,csr_mon,csr_leap,csr_dec,valid_rate
0,random,val,0.088600,0.145800,1.000000,0.628200,1.000000,1.000000
1,smart_random,val,0.999800,0.999800,1.000000,0.999800,1.000000,1.000000
2,random,test_random,0.096200,0.145400,1.000000,0.628800,1.000000,1.000000
3,smart_random,test_random,0.999600,0.999600,1.000000,0.999600,1.000000,1.000000
4,random,held_out_tuples,0.107623,0.157848,1.000000,0.652915,1.000000,1.000000
5,smart_random,held_out_tuples,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,cvae,val,0.999000,0.999000,0.999800,0.999800,0.999800,0.999800
7,cvae,test_random,0.999200,0.999200,0.999800,0.999800,0.999800,0.999800
8,cvae,held_out_tuples,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,acgan,val,0.149400,0.149600,0.998200,0.996000,0.998200,0.998200


In [20]:
!python model/predict.py -i data/example_input.txt -o predictions.txt --model cvae
!head -3 predictions.txt && echo --- && wc -l predictions.txt


[predict.py] model=cvae  wrote 1465 predictions -> predictions.txt
[WED] [JAN] [False] [180] 28-1-1801
[MON] [JAN] [False] [190] 26-1-1903
[SAT] [JAN] [True] [200] 12-1-2008
---
1465 predictions.txt


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0     62      0 --:--:-- --:--:-- --:--:--    62
404: Not Found---
0 predictions.txt


## 8. Download artefacts from Colab

Bundles predictions, weights, results CSVs and figures into a zip you
can save locally. Works in Colab; no-ops in local Jupyter.


In [ ]:
# Download predictions + weights + results from Colab.
import os, zipfile, glob
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    bundle = 'dates-generator-outputs.zip'
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in (['predictions.txt']
                     + glob.glob('model/weights/*.pt')
                     + glob.glob('model/weights/*.json')
                     + glob.glob('results/*.csv')
                     + glob.glob('results/*.json')
                     + glob.glob('results/figures/*.png')):
            if os.path.exists(path):
                z.write(path)
                print('  +', path)
    print(f'\nwrote {bundle} ({os.path.getsize(bundle)/1e6:.1f} MB)')
    from google.colab import files
    files.download(bundle)
    # If you only need predictions.txt by itself:
    # files.download('predictions.txt')
else:
    print('Not in Colab; everything is already on your local disk.')
